In [1]:
import random
import numpy as np

random.seed(42)
np.random.seed(42)

import pandas as pd

import matplotlib.pyplot as plt

In [2]:
embeddings_s = np.load('../Model/emb_s_all-MiniLM-L6-v2.npy')

description_text_s = pd.read_csv('../Output/london_app_s_tokens.csv')

In [3]:
import ast

description_text_s['tokens'] = description_text_s['tokens'].apply(ast.literal_eval)

print(type(description_text_s['tokens'].iloc[0]))

<class 'list'>


In [4]:
docs_s = description_text_s['tokens'].apply(lambda tokens: ' '.join(tokens)).tolist()

In [5]:
def compute_diversity_score(topic_model):
    """
    Compute topic diversity score as the ratio of unique keywords 
    to total keywords across all topics (excluding topic -1).
    """
    topics_ = topic_model.get_topics()
    all_words = [
        word for topic_id, words in topics_.items() if topic_id != -1
        for word, _ in words
    ]
    unique_words = set(all_words)
    return len(unique_words) / len(all_words) if all_words else 0

In [6]:
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

def compute_coherence_score(topic_model, docs, top_n_words=10):

    # Tokenize documents
    tokenized_docs = [doc.split() for doc in docs if isinstance(doc, str) and doc.strip()]
    
    # Build gensim dictionary
    dictionary = Dictionary(tokenized_docs)

    # Restrict to words in vectorizer vocabulary
    vectorizer = topic_model.vectorizer_model
    vocab = set(vectorizer.get_feature_names_out())

    topic_words = []
    topic_ids = topic_model.get_topic_info().Topic.tolist()
    for topic_id in topic_ids:
        if topic_id == -1:
            continue
        words = [w for w, _ in topic_model.get_topic(topic_id)[:top_n_words]]
        filtered = [w for w in words if w in vocab and w in dictionary.token2id]
        if filtered:
            topic_words.append(filtered)

    # Compute coherence
    coherence_model = CoherenceModel(
        topics=topic_words,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence='c_v'
    )
    return coherence_model.get_coherence()

In [7]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

In [8]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
import itertools

param_grid = {
    "n_neighbors": [10, 15, 30],
    "min_dist": [0.0, 0.1, 0.3],
    "min_cluster_size": [10, 15, 30],
    "min_samples": [5, 10, 15],
    "ngram_range": [(1, 1), (1, 2)]
}

results = []

for nn, dist, mcs, ms, ngram in itertools.product(*param_grid.values()):

    print(f"Testing: n_neighbors={nn}, min_dist={dist}, min_cluster_size={mcs}, min_samples={ms}, ngram_range={ngram}")

    umap_model = UMAP(
        n_neighbors=nn, 
        n_components=5, 
        min_dist=dist, 
        metric='cosine'
    )
    
    hdbscan_model = HDBSCAN(
        min_cluster_size=mcs,
        min_samples=ms,
        prediction_data=True
    )
    
    vectorizer_model = CountVectorizer(ngram_range=ngram)

    topic_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        calculate_probabilities=False,
        verbose=False,
        top_n_words=10
    )

    topics, _ = topic_model.fit_transform(docs_s, embeddings=embeddings_s)
    coherence = compute_coherence_score(topic_model, docs_s)
    diversity = compute_diversity_score(topic_model)
    outlier_ratio = sum([1 for t in topics if t == -1]) / len(topics)

    results.append({
        'n_neighbors': nn,
        'min_dist': dist,
        'min_cluster_size': mcs,
        'min_samples': ms,
        'ngram_range': str(ngram),
        'coherence': coherence,
        'diversity': diversity,
        'outlier_ratio': outlier_ratio
    })

Testing: n_neighbors=10, min_dist=0.0, min_cluster_size=10, min_samples=5, ngram_range=(1, 1)
Testing: n_neighbors=10, min_dist=0.0, min_cluster_size=10, min_samples=5, ngram_range=(1, 2)
Testing: n_neighbors=10, min_dist=0.0, min_cluster_size=10, min_samples=10, ngram_range=(1, 1)
Testing: n_neighbors=10, min_dist=0.0, min_cluster_size=10, min_samples=10, ngram_range=(1, 2)
Testing: n_neighbors=10, min_dist=0.0, min_cluster_size=10, min_samples=15, ngram_range=(1, 1)
Testing: n_neighbors=10, min_dist=0.0, min_cluster_size=10, min_samples=15, ngram_range=(1, 2)
Testing: n_neighbors=10, min_dist=0.0, min_cluster_size=15, min_samples=5, ngram_range=(1, 1)
Testing: n_neighbors=10, min_dist=0.0, min_cluster_size=15, min_samples=5, ngram_range=(1, 2)
Testing: n_neighbors=10, min_dist=0.0, min_cluster_size=15, min_samples=10, ngram_range=(1, 1)
Testing: n_neighbors=10, min_dist=0.0, min_cluster_size=15, min_samples=10, ngram_range=(1, 2)
Testing: n_neighbors=10, min_dist=0.0, min_cluster_siz

In [1]:
results

NameError: name 'results' is not defined

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

df_results = pd.DataFrame(results)

fig = plt.figure(figsize=(10, 6))

ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(df_results['coherence'], df_results['diversity'], df_results['outlier_ratio'],
                c=df_results['coherence'], cmap='viridis', s=60)
ax.set_xlabel('Coherence (↑)')
ax.set_ylabel('Diversity (↑)')
ax.set_zlabel('Outlier Ratio (↓)')

plt.title("BERTopic Hyperparameter Tuning Results")
plt.colorbar(sc, label='Coherence Score')
plt.tight_layout()
plt.show()